# 13. YOLO 핵심 아이디어

이 노트북은 YOLO가 왜 빠른 one-stage detector인지 직관적으로 이해하는 데 초점을 둡니다.

YOLO는 이미지를 grid로 나누고, 각 grid cell이 자신이 담당하는 객체의 bounding box와 class score를 예측합니다. 즉, 후보 영역을 따로 만들고 다시 분류하는 대신, 한 번의 forward pass에서 dense prediction을 수행합니다.

이번 노트북의 목표는 다음과 같습니다.

- YOLO가 이미지를 grid로 나누는 이유를 이해합니다.
- 객체 중심점이 어떤 cell에 속하는지 계산합니다.
- grid cell이 예측하는 값의 구성을 단순화해서 확인합니다.
- confidence, class probability, final score의 관계를 이해합니다.
- 다음 노트북의 YOLO 출력 해석으로 넘어갈 준비를 합니다.


In [ ]:
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

available_fonts = {f.name for f in fm.fontManager.ttflist}
for font_name in ['Malgun Gothic', 'AppleGothic', 'NanumGothic']:
    if font_name in available_fonts:
        plt.rcParams['font.family'] = font_name
        break

plt.rcParams['figure.figsize'] = (7, 5)
plt.rcParams['axes.unicode_minus'] = False


## 13-1. YOLO의 큰 생각

YOLO는 이미지를 여러 후보로 잘라서 하나씩 분류하지 않습니다. 대신 이미지 전체를 한 번 보고, feature map의 각 위치에서 박스와 클래스를 동시에 예측합니다.

초기 YOLO 설명에서는 이미지를 `S x S` grid로 나누고, 각 grid cell이 객체 중심점을 포함하면 그 객체를 담당한다고 설명합니다.

이 방식의 장점은 명확합니다.

- 이미지 전체 문맥을 한 번에 봅니다.
- proposal 단계를 따로 두지 않습니다.
- 많은 위치의 예측을 병렬로 계산할 수 있습니다.
- 실시간 탐지에 유리합니다.


## 13-2. Grid로 이미지 나누기

아래 예시는 `240 x 180` 이미지를 `4 x 3` grid로 나눈 것입니다. 실제 YOLO 모델은 입력 크기와 feature map 해상도에 따라 더 많은 grid cell을 사용합니다.


In [ ]:
image_width, image_height = 240, 180
grid_cols, grid_rows = 4, 3
cell_w = image_width / grid_cols
cell_h = image_height / grid_rows

objects = [
    {'class': 'cat', 'box': (45, 35, 125, 115)},
    {'class': 'dog', 'box': (150, 75, 220, 155)},
]

def box_center(box):
    x1, y1, x2, y2 = box
    return ((x1 + x2) / 2, (y1 + y2) / 2)


def responsible_cell(center):
    cx, cy = center
    col = min(grid_cols - 1, int(cx // cell_w))
    row = min(grid_rows - 1, int(cy // cell_h))
    return row, col


fig, ax = plt.subplots(figsize=(7, 5))
ax.set_xlim(0, image_width)
ax.set_ylim(image_height, 0)
ax.set_facecolor('#f8fafc')
ax.set_title('YOLO grid와 객체 중심점')
ax.set_xticks([])
ax.set_yticks([])

for col in range(grid_cols + 1):
    ax.axvline(col * cell_w, color='gray', linewidth=1, alpha=0.7)
for row in range(grid_rows + 1):
    ax.axhline(row * cell_h, color='gray', linewidth=1, alpha=0.7)

colors = {'cat': 'crimson', 'dog': 'royalblue'}
for obj in objects:
    x1, y1, x2, y2 = obj['box']
    color = colors[obj['class']]
    cx, cy = box_center(obj['box'])
    row, col = responsible_cell((cx, cy))
    ax.add_patch(Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor=color, linewidth=2.5))
    ax.scatter([cx], [cy], color=color, s=60)
    ax.text(x1, y1 - 5, f"{obj['class']} -> cell({row}, {col})", color=color, fontsize=10, weight='bold')

plt.show()

for obj in objects:
    center = box_center(obj['box'])
    print(obj['class'], 'center=', center, 'responsible_cell=', responsible_cell(center))


## 13-3. Cell 내부 상대 좌표

YOLO는 객체 중심점의 위치를 이미지 전체 좌표 그대로만 쓰지 않고, 담당 cell 안에서의 상대 위치로 표현할 수 있습니다.

예를 들어 어떤 객체 중심이 cell의 왼쪽 위에 가까우면 `(0.1, 0.2)`처럼 표현되고, cell 중앙에 있으면 `(0.5, 0.5)` 근처가 됩니다.

이렇게 하면 각 cell이 자기 영역 안에서 객체 중심이 어디에 있는지만 예측하면 됩니다.


In [ ]:
def center_to_cell_relative(center):
    cx, cy = center
    row, col = responsible_cell(center)
    rel_x = (cx - col * cell_w) / cell_w
    rel_y = (cy - row * cell_h) / cell_h
    return row, col, rel_x, rel_y


for obj in objects:
    row, col, rel_x, rel_y = center_to_cell_relative(box_center(obj['box']))
    print(f"{obj['class']}: cell=({row}, {col}), rel_x={rel_x:.3f}, rel_y={rel_y:.3f}")


## 13-4. 하나의 cell이 예측하는 값

단순화하면 한 grid cell은 다음 정보를 예측한다고 볼 수 있습니다.

- box 좌표: `tx, ty, tw, th`
- objectness 또는 confidence: 이 cell/box에 객체가 있을 가능성
- class probability: 객체가 각 클래스일 확률

실제 YOLO 버전마다 anchor, scale, activation, output decoding 방식은 다르지만, 핵심은 `박스 위치 + 객체 존재 여부 + 클래스`를 함께 예측한다는 점입니다.


In [ ]:
classes = ['cat', 'dog', 'car']

cell_prediction = {
    'cell': (0, 1),
    'box_cxcywh_norm': (0.42, 0.58, 0.33, 0.44),
    'objectness': 0.90,
    'class_probs': {'cat': 0.82, 'dog': 0.12, 'car': 0.06},
}

print('cell:', cell_prediction['cell'])
print('box:', cell_prediction['box_cxcywh_norm'])
print('objectness:', cell_prediction['objectness'])
print('class_probs:', cell_prediction['class_probs'])


## 13-5. Final score 계산

모델이 `objectness`와 클래스 확률을 따로 출력한다면, 특정 클래스에 대한 최종 점수는 보통 다음처럼 볼 수 있습니다.

$$score(class) = objectness \times P(class)$$

즉, 박스 안에 객체가 있을 가능성과 그 객체가 특정 클래스일 가능성을 함께 고려합니다.


In [ ]:
objectness = cell_prediction['objectness']
final_scores = {
    class_name: objectness * prob
    for class_name, prob in cell_prediction['class_probs'].items()
}

for class_name, score in sorted(final_scores.items(), key=lambda item: item[1], reverse=True):
    print(f'{class_name:<3}: final score={score:.3f}')

best_class = max(final_scores, key=final_scores.get)
print('best class:', best_class)


## 13-6. Dense prediction으로 보기

YOLO는 하나의 cell만 예측하지 않습니다. 모든 grid cell이 동시에 예측합니다. 대부분의 cell에는 객체가 없으므로 objectness가 낮아야 하고, 객체 중심이 들어 있는 cell은 높은 objectness를 가져야 합니다.

아래 예시는 grid별 objectness score를 단순하게 만든 것입니다.


In [ ]:
objectness_grid = [
    [0.05, 0.88, 0.12, 0.03],
    [0.04, 0.20, 0.16, 0.77],
    [0.02, 0.08, 0.11, 0.18],
]

fig, ax = plt.subplots(figsize=(7, 4))
im = ax.imshow(objectness_grid, cmap='YlOrRd', vmin=0, vmax=1)
ax.set_title('Grid별 objectness score')
ax.set_xlabel('grid col')
ax.set_ylabel('grid row')

for row in range(grid_rows):
    for col in range(grid_cols):
        ax.text(col, row, f'{objectness_grid[row][col]:.2f}', ha='center', va='center', color='black', weight='bold')

plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.show()


## 13-7. 여러 박스를 만들고 threshold 적용하기

실제 YOLO 출력에서는 많은 후보 박스가 생성됩니다. 그중 final score가 낮은 박스를 먼저 버리고, 남은 박스에 NMS를 적용합니다.

흐름은 다음과 같습니다.

1. grid와 box별 raw output을 박스 좌표와 score로 변환합니다.
2. score threshold보다 낮은 후보를 제거합니다.
3. 클래스별 NMS를 적용합니다.
4. 최종 탐지 결과만 남깁니다.


In [ ]:
candidate_boxes = [
    {'class': 'cat', 'box': (45, 35, 125, 115), 'score': 0.74},
    {'class': 'cat', 'box': (50, 40, 130, 120), 'score': 0.61},
    {'class': 'dog', 'box': (150, 75, 220, 155), 'score': 0.69},
    {'class': 'car', 'box': (10, 120, 70, 170), 'score': 0.18},
]

score_threshold = 0.5
kept_by_score = [pred for pred in candidate_boxes if pred['score'] >= score_threshold]

print('score threshold:', score_threshold)
for pred in kept_by_score:
    print(pred)

fig, ax = plt.subplots(figsize=(7, 5))
ax.set_xlim(0, image_width)
ax.set_ylim(image_height, 0)
ax.set_facecolor('#f8fafc')
ax.set_title('Score threshold 통과 후보')
ax.set_xticks([])
ax.set_yticks([])

color_map = {'cat': 'crimson', 'dog': 'royalblue', 'car': 'seagreen'}
for pred in kept_by_score:
    x1, y1, x2, y2 = pred['box']
    color = color_map[pred['class']]
    ax.add_patch(Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor=color, linewidth=2.5))
    ax.text(x1, y1 - 5, f"{pred['class']} {pred['score']:.2f}", color=color, fontsize=10, weight='bold')

plt.show()


## 13-8. YOLO가 빠른 이유

YOLO가 빠른 이유는 단순히 모델이 작아서만은 아닙니다. 문제를 푸는 방식 자체가 빠른 구조입니다.

- 이미지를 한 번의 forward pass로 처리합니다.
- proposal을 만들고 각 후보를 다시 분류하는 단계를 줄입니다.
- grid 전체의 예측을 convolution 연산으로 병렬 계산합니다.
- 후처리는 thresholding과 NMS 중심으로 단순합니다.

물론 모든 장면에서 무조건 좋은 것은 아닙니다. 작은 객체가 많거나 서로 매우 가까운 객체가 많으면 grid 기반 예측이 어려울 수 있습니다. 최신 YOLO 계열은 이런 문제를 줄이기 위해 multi-scale feature와 anchor-free/head 개선 등 다양한 구조를 사용합니다.


## 정리

- YOLO는 이미지를 grid로 나누고 각 위치에서 박스와 클래스를 동시에 예측합니다.
- 객체 중심점이 들어 있는 cell이 그 객체를 담당한다고 이해할 수 있습니다.
- 각 예측은 박스 좌표, objectness, class probability를 포함합니다.
- 최종 class score는 objectness와 class probability를 함께 고려합니다.
- 많은 후보 박스는 score threshold와 NMS를 거쳐 최종 탐지 결과로 정리됩니다.

다음 노트북 `14_YOLO_출력_해석과_추론.ipynb`에서는 YOLO 출력 텐서를 박스, 클래스, confidence로 해석하고 후처리하는 과정을 더 구체적으로 다룹니다.
